# Feature Selection Check

In this notebook, we will be checking what feature selection technique would be better for us to use. We mainly want to distinguish between pearson, forward selection and Recursive Feature Elimination. We will explore each one about looking at the f1 scores each provides for a logistic regresssion.

## Setting Up

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.feature_selection import SequentialFeatureSelector

# 1. Reading the data set

df= pd.read_csv('passenger_survey_balanced.csv')
df.head()

# 2. Cleaning the data set

df = df.drop_duplicates()
sensitive = ['gender', 'age_group', 'nationality', 'education', 'household_income',
             'has_disability', 'uses_assistive_device', 'requested_special_assistance']
X = df.drop(columns=['liked'] + sensitive)
y = df['liked']

# 3. Training / validation split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 4. Preprocess 

num_cols = X.select_dtypes('number').columns
cat_cols = X.select_dtypes(exclude='number').columns

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=20,
                                           sparse_output=False))]), cat_cols),])
Xtr = prep.fit_transform(X_train)
Xva = prep.transform(X_val)
names = prep.get_feature_names_out()
print("Features after encoding:", Xtr.shape[1])

# 5. Setting up check for which is bettter
def score_cols(cols):
    clf = LogisticRegression(max_iter=1500).fit(Xtr[:, cols], y_train)
    return f1_score(y_val, clf.predict(Xva[:, cols]), average='macro')

ks = [5, 10, 20, 40, 80, 160]
results = {}


Features after encoding: 212


We increase the number of features after encoding, but want to decrease overall, which is why we are using feature selection to help us determine the best features.

## Pearson

In [2]:
corr = np.nan_to_num([abs(np.corrcoef(Xtr[:, i], y_train)[0, 1]) for i in range(Xtr.shape[1])])
pearson_order = np.argsort(-corr)
results['pearson'] = [score_cols(pearson_order[:k]) for k in ks]

Above we perform pearson so that it checks the coreelation between the y and the x each time from our training values.

## Forward selection

In [3]:
pool = list(pearson_order[:40])
rng = np.random.RandomState(42)
sub = rng.choice(len(Xtr), size=min(15000, len(Xtr)), replace=False)
Xa, Xb, ya, yb = train_test_split(Xtr[sub], y_train.iloc[sub], test_size=0.25,
                                  stratify=y_train.iloc[sub], random_state=42)

# Greedy forward selection, run ONCE; the order it picks gives every k
selected, remaining = [], pool.copy()
for step in range(20):
    best_feat, best_score = None, -1
    for f in remaining:
        cols = selected + [f]
        clf = LogisticRegression(max_iter=300).fit(Xa[:, cols], ya)
        s = f1_score(yb, clf.predict(Xb[:, cols]), average='macro')
        if s > best_score:
            best_feat, best_score = f, s
    selected.append(best_feat)
    remaining.remove(best_feat)
    print(f"step {step+1}: added {names[best_feat]} ({best_score:.4f})")

# Evaluate on the real validation set with the full training data
fwd_ks = [5, 10, 20]
results['forward'] = [score_cols(selected[:k]) for k in fwd_ks] + [np.nan] * 3

step 1: added num__overall_airport_cleanliness (0.6743)
step 2: added num__location_and_movement (0.7231)
step 3: added num__boarding_lounge_comfort (0.7440)
step 4: added num__baggage_carousel_identification_ease_is_applicable (0.7926)
step 5: added num__food_beverage_outlets (0.7984)
step 6: added num__security_screening_process (0.8000)
step 7: added num__restroom_cleanliness_is_applicable (0.8037)
step 8: added cat__disembarkation_method_used_missing (0.8053)
step 9: added num__restroom_quantity_is_applicable (0.8053)
step 10: added num__restroom_maintenance_is_applicable (0.8053)
step 11: added num__baggage_claim_time_is_applicable (0.8053)
step 12: added num__baggage_integrity_is_applicable (0.8053)
step 13: added cat__connection_Boarded at the airport (0.8053)
step 14: added num__airport_internet (0.8055)
step 15: added num__power_outlet_availability (0.8062)
step 16: added num__acoustic_comfort_is_applicable (0.8078)
step 17: added num__retail_outlets (0.8090)
step 18: added nu

It takes to long to run when we put it past 20 steps, so we will not be susing this value. Also, seems to be lower than other as we go towards 80.

## RFE

In [4]:
rfe = RFE(LogisticRegression(max_iter=1500), n_features_to_select=1, step=5).fit(Xtr, y_train)
rfe_order = np.argsort(rfe.ranking_)
results['rfe'] = [score_cols(rfe_order[:k]) for k in ks]

Above we perform RFE in order to see the best features.

## Results

In [5]:
print(pd.DataFrame(results, index=ks).round(4))

     pearson  forward     rfe
5     0.7391   0.7990  0.7726
10    0.7528   0.7972  0.7877
20    0.8066   0.8007  0.8048
40    0.8112      NaN  0.8114
80    0.8211      NaN  0.8248
160   0.8255      NaN  0.8250


From the table above, we can see that RFE and pearson are both solid picks. Since RFE is widely considered better, we will use RFE as the reason for method selection for our graph. Also, since we barely increase from 80 to 160, it is better to have 80 be the number of features we use for RFE. This means that we want from having 140 features to having 80.